<a href="https://colab.research.google.com/github/jacobhroutzong/coding-challenges/blob/main/tensara/1D_Convolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:


def conv1d(A,B):
  N = len(A)
  K = len(B)
  r =  (K-1) // 2

  C = [0.0] * N # builds an array C

  for i in range(N): # i -> arr N
    total = 0.0 # initializes target accumulator

    for j in range(K): # i -> arr K
      index = i + j - r # calculates radius

      if 0 <= index < N: # checks radius for out of bounds error
        total += A[index] * B[j]
        # accumulates A[index] * B[j]
    C[i] = total
    # adds value in index

  return C







In [ ]:

import torch
import triton
import triton.language as tl

@triton.jit
def conv1d_kernel(A, B, C,
                  N: tl.constexpr,
                  K: tl.constexpr,
                  BLOCK: tl.constexpr): # A = input array, B = weights, C = Output array
                                        # N = len(A), K = len(B), BLOCK = number of pos handled by Triton

    i = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    # initializes i as a tensor of 128 indices
    # * program_id

    ### program_id is a tritonic function equal to the launch/execution grid
    ### found by triton.cdic(N, BLOCK) which is equal to len(A) / BLOCK
    ### important for triton

    r = (K - 1) // 2  # finds radius

    total = tl.full((BLOCK,), 0.0, tl.float32)
    # initializes a tensor accumulator of shape (128, ) type float(32)


    for j in range(K): # finds j by iterating through K (len(B))
        index = i + j - r
        #  i (an array) + j - r
        A_values = tl.load(
            A + index, # tensor addition
            mask=(i < N) & (index >= 0) & (index < N), # oob check
            other=0.0
        )

        B_value = tl.load(B + j) # tensor addition

        total += A_values * B_value
        # updates accumulator for A[index] * B[j]

    tl.store(C + i, total, mask=i < N)
    # stores final total tensor at program_id associated locations



def triton_conv1d(A, B):
    N = len(A)  # number of elements in A
    K = len(B)  # number of elements in B

    C = torch.empty_like(A)  # builds an array C

    BLOCK = 128
    grid = (triton.cdiv(N, BLOCK),)

    conv1d_kernel[grid](A, B, C, N, K, BLOCK)

    return C


In [9]:

import torch
import torch.nn.functional as F

# Creates input arrays
A = torch.tensor([3, 6, 9, 12, 15, 18, 21], device="cuda", dtype=torch.float32)
B = torch.tensor([2, 1, 3], device="cuda", dtype=torch.float32)

N = len(A)
K = len(B)
r = (K - 1) // 2

# Runs Triton convolution
C = triton_conv1d(A, B)

# Runs PyTorch convolution
C_pytorch = F.conv1d(
    A.reshape(1, 1, N),
    B.reshape(1, 1, K),
    padding=r
).flatten()

# Compares results
print("Triton:", C.tolist())
print("PyTorch:", C_pytorch.tolist())

torch.testing.assert_close(C, C_pytorch)
print("PASS!")


Triton: [21.0, 39.0, 57.0, 75.0, 93.0, 111.0, 57.0]
PyTorch: [21.0, 39.0, 57.0, 75.0, 93.0, 111.0, 57.0]
PASS!
